# Netflix Titles: Exploratory Data Analysis (Phase 2)
Uses the files produced by `data_cleaner.ipynb`:

| File | Grain |
|---|---|
| `netflix_titles_cleaned.csv` | one row per title |
| `genres_long.csv` | one row per title and genre |
| `countries_long.csv` | one row per title and country |

**Goal:** decide which charts earn a place in the Streamlit dashboard and collect 5-8 headline findings. Each section ends with a *Dashboard note* saying where the chart would go.

Sections: 1 Setup, 2 Genre popularity, 3 Release-year distribution, 4 Regional patterns, 5 Extras (duration, ratings, timing), 6 Headline findings.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

RED, BLACK, GREY = "#E50914", "#221F1F", "#B3B3B3"
TYPE_COLORS = {"Movie": RED, "TV Show": BLACK}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"

In [ ]:
df = pd.read_csv("../data/netflix_titles_cleaned.csv", parse_dates=["date_added"])
genres = pd.read_csv("../data/genres_long.csv")
countries = pd.read_csv("../data/countries_long.csv")

print("Titles:       ", df.shape)
print("Genre rows:   ", genres.shape)
print("Country rows: ", countries.shape)
df.head(3)

Quick sanity check on the loaded data before analysing it.

In [ ]:
print(df["type"].value_counts(), "\n")
print("Missing date_added:", df["date_added"].isna().sum())
print("Release years:", df["release_year"].min(), "to", df["release_year"].max())
print("Year added:   ", int(df["year_added"].min()), "to", int(df["year_added"].max()))

## 2. Genre popularity
Genres come from the exploded `genres_long.csv`. A title with three genres counts once in each, so genre totals add up to more than the number of titles.

### 2.1 Top genres overall

In [ ]:
top_genres = genres["genre"].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, color=RED)
plt.title("Top 15 Genres on Netflix")
plt.xlabel("Number of titles")
plt.ylabel("")
plt.tight_layout()
plt.show()

top_genres.head(5)

### 2.2 Top genres by content type
Movie and TV genre labels are different (e.g. `Dramas` vs `TV Dramas`), so each type gets its own chart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, t in zip(axes, ["Movie", "TV Show"]):
    top = genres.loc[genres["type"] == t, "genre"].value_counts().head(10)
    sns.barplot(x=top.values, y=top.index, color=TYPE_COLORS[t], ax=ax)
    ax.set_title(f"Top 10 {t} Genres")
    ax.set_xlabel("Number of titles")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

### 2.3 Genre popularity over time (by year added)
Shows how the top genres' share of new additions changed. Years with very few titles are left out to avoid noisy lines.

In [ ]:
g = genres.dropna(subset=["year_added"]).copy()
g["year_added"] = g["year_added"].astype(int)

top5 = g["genre"].value_counts().head(5).index
yearly = g.groupby("year_added").size()
valid_years = yearly[yearly >= 50].index

trend = (g[g["genre"].isin(top5) & g["year_added"].isin(valid_years)]
         .groupby(["year_added", "genre"]).size().unstack(fill_value=0))

trend.plot(figsize=(11, 5), marker="o")
plt.title("Titles Added per Year: Top 5 Genres")
plt.xlabel("Year added")
plt.ylabel("Number of titles")
plt.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 2.4 Genre share of additions by year (percentage)

In [ ]:
share = trend.div(trend.sum(axis=1), axis=0) * 100

share.plot(kind="area", stacked=True, figsize=(11, 5), alpha=0.85)
plt.title("Share of Top-5 Genre Additions by Year (%)")
plt.xlabel("Year added")
plt.ylabel("Share (%)")
plt.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Dashboard note:** Genres tab: (a) top-N bar chart with a Movie / TV Show toggle (2.1, 2.2), (b) trend line for genres chosen in the sidebar (2.3). The share chart (2.4) is optional.

## 3. Release-year distribution
`release_year` is when a title was made; `year_added` is when it arrived on Netflix.

### 3.1 Titles by release year

In [ ]:
plt.figure(figsize=(11, 5))
sns.histplot(data=df, x="release_year", hue="type", multiple="stack", binwidth=2, palette=TYPE_COLORS)
plt.title("Titles by Release Year")
plt.xlabel("Release year")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

### 3.2 Titles by release decade

In [ ]:
decade_counts = (df.groupby(["release_decade", "type"]).size().unstack(fill_value=0)
                   .sort_index())

decade_counts.plot(kind="bar", stacked=True, color=[TYPE_COLORS[c] for c in decade_counts.columns], figsize=(10, 5))
plt.title("Titles by Release Decade")
plt.xlabel("Release decade")
plt.ylabel("Number of titles")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.3 Released vs added to Netflix
Two lines on the same axis: when titles were made and when they arrived.

In [ ]:
since = 2008
released = df[df["release_year"] >= since].groupby("release_year").size()
added = df.dropna(subset=["year_added"]).groupby(df["year_added"].astype("Int64")).size()
added = added[added.index >= since]

plt.figure(figsize=(11, 5))
plt.plot(released.index, released.values, marker="o", color=BLACK, label="Released")
plt.plot(added.index.astype(int), added.values, marker="o", color=RED, label="Added to Netflix")
plt.title(f"Titles Released vs Added to Netflix ({since} onwards)")
plt.xlabel("Year")
plt.ylabel("Number of titles")
plt.legend()
plt.tight_layout()
plt.show()

### 3.4 Release lag
`release_lag = year_added - release_year`. A lag of 0 means the title was added in its release year. Negative values mean the title was added *before* its release year, and are probably data issues, so they are excluded from the charts below.

In [ ]:
lag = df.dropna(subset=["release_lag"]).copy()
lag["release_lag"] = lag["release_lag"].astype(int)

print("Negative lag (added before release year):", (lag["release_lag"] < 0).sum())
lag = lag[lag["release_lag"] >= 0]

plt.figure(figsize=(10, 5))
sns.histplot(data=lag, x="release_lag", hue="type", multiple="stack", binrange=(0, 20), binwidth=1, palette=TYPE_COLORS)
plt.title("Years Between Release and Arrival on Netflix")
plt.xlabel("Release lag (years, capped at 20 on the chart)")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

print(lag.groupby("type")["release_lag"].agg(["median", "mean"]).round(1))

### 3.5 Median release lag by year added
A falling lag suggests Netflix shifted toward newer content.

In [ ]:
lag_by_year = lag.groupby("year_added")["release_lag"].median()
lag_by_year = lag_by_year[lag.groupby("year_added").size() >= 50]

plt.figure(figsize=(10, 4.5))
plt.plot(lag_by_year.index.astype(int), lag_by_year.values, marker="o", color=RED)
plt.title("Median Release Lag by Year Added")
plt.xlabel("Year added")
plt.ylabel("Median lag (years)")
plt.tight_layout()
plt.show()

**Dashboard note:** Timeline tab: (a) release-year histogram or decade bar (3.1, 3.2), (b) released-vs-added line chart (3.3), (c) release-lag KPI plus histogram (3.4). Respect the year-range sidebar filter.

## 4. Regional patterns
Uses `countries_long.csv`. A co-produced title counts once for each country. Titles with an `Unknown` country are excluded from the rankings.

In [ ]:
known = countries[countries["country_single"] != "Unknown"]
print("Distinct countries:", known["country_single"].nunique())
print("Titles with unknown country:", (countries.groupby("show_id")["country_single"].agg(lambda s: (s == "Unknown").all())).sum())

### 4.1 Top 15 countries

In [ ]:
top_countries = known["country_single"].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, color=RED)
plt.title("Top 15 Content-Producing Countries")
plt.xlabel("Number of titles")
plt.ylabel("")
plt.tight_layout()
plt.show()

print((top_countries.head(5) / df.shape[0] * 100).round(1).rename("% of all titles"))

### 4.2 Movies vs TV shows in the top 10 countries

In [ ]:
top10 = top_countries.head(10).index
mix = (known[known["country_single"].isin(top10)]
       .groupby(["country_single", "type"]).size().unstack(fill_value=0)
       .loc[top10])

mix.plot(kind="barh", stacked=True, color=[TYPE_COLORS[c] for c in mix.columns], figsize=(10, 5.5))
plt.gca().invert_yaxis()
plt.title("Movies vs TV Shows by Country (Top 10)")
plt.xlabel("Number of titles")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 4.3 Country by genre
Share of each country's titles that fall into each genre (rows sum to 100%), for the top 8 countries and top 10 genres.

In [ ]:
cg = known.merge(genres[["show_id", "genre"]], on="show_id")

top8 = known["country_single"].value_counts().head(8).index
top10_genres = genres["genre"].value_counts().head(10).index

table = (cg[cg["country_single"].isin(top8) & cg["genre"].isin(top10_genres)]
         .groupby(["country_single", "genre"]).size().unstack(fill_value=0)
         .loc[top8, top10_genres])
table_pct = table.div(table.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 5.5))
sns.heatmap(table_pct, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% of country's genre tags"})
plt.title("Genre Mix by Country (% within country)")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

### 4.4 Share of non-US content over time
A title counts as *US-involved* if the United States appears among its countries. Titles with an unknown country are excluded.

In [ ]:
us_flag = (countries.groupby("show_id")["country_single"]
           .agg(us=lambda s: (s == "United States").any(), unknown=lambda s: (s == "Unknown").all()))

d = df.merge(us_flag, left_on="show_id", right_index=True)
d = d[~d["unknown"]].dropna(subset=["year_added"]).copy()
d["year_added"] = d["year_added"].astype(int)

by_year = d.groupby("year_added")["us"].agg(["mean", "size"])
by_year = by_year[by_year["size"] >= 50]
non_us_share = (1 - by_year["mean"]) * 100

plt.figure(figsize=(10, 4.5))
plt.plot(non_us_share.index, non_us_share.values, marker="o", color=RED)
plt.title("Share of Non-US Titles by Year Added (%)")
plt.xlabel("Year added")
plt.ylabel("% of titles without a US credit")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

non_us_share.round(1)

### 4.5 Top countries over time (by year added)

In [ ]:
kc = known.dropna(subset=["year_added"]).copy()
kc["year_added"] = kc["year_added"].astype(int)

top5_c = known["country_single"].value_counts().head(5).index
yr_totals = kc.groupby("year_added").size()
ok_years = yr_totals[yr_totals >= 50].index

ctrend = (kc[kc["country_single"].isin(top5_c) & kc["year_added"].isin(ok_years)]
          .groupby(["year_added", "country_single"]).size().unstack(fill_value=0))

ctrend.plot(figsize=(11, 5), marker="o")
plt.title("Titles Added per Year: Top 5 Countries")
plt.xlabel("Year added")
plt.ylabel("Number of titles")
plt.legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Dashboard note:** Regions tab: (a) top-N country bar (4.1), (b) world choropleth built from the country counts (Plotly is better than Matplotlib here), (c) country-by-genre heatmap (4.3), (d) non-US share line (4.4).

## 5. Extras: duration, ratings and timing

### 5.1 Movie duration distribution

In [ ]:
movies = df[df["type"] == "Movie"].dropna(subset=["movie_minutes"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.histplot(movies["movie_minutes"], bins=30, color=RED, ax=axes[0])
axes[0].set_title("Movie Duration Distribution")
axes[0].set_xlabel("Minutes")
axes[0].set_ylabel("Number of movies")
axes[0].axvline(movies["movie_minutes"].median(), color=BLACK, linestyle="--", label="Median")
axes[0].legend()

sns.boxplot(y=movies["movie_minutes"], color=RED, ax=axes[1])
axes[1].set_title("Spread and Outliers")
axes[1].set_ylabel("Minutes")

plt.tight_layout()
plt.show()

print(movies["movie_minutes"].describe().round(1))

### 5.2 Seasons per TV show

In [ ]:
shows = df[df["type"] == "TV Show"].dropna(subset=["tv_seasons"]).copy()
shows["tv_seasons"] = shows["tv_seasons"].astype(int)
season_counts = shows["tv_seasons"].clip(upper=8).value_counts().sort_index()
labels = [str(i) if i < 8 else "8+" for i in season_counts.index]

plt.figure(figsize=(9, 4.5))
sns.barplot(x=labels, y=season_counts.values, color=BLACK)
plt.title("TV Shows by Number of Seasons")
plt.xlabel("Seasons")
plt.ylabel("Number of shows")
plt.tight_layout()
plt.show()

print("Share of shows with a single season:", round((shows["tv_seasons"] == 1).mean() * 100, 1), "%")

### 5.3 Ratings breakdown

In [ ]:
rating_counts = df.groupby(["rating", "type"]).size().unstack(fill_value=0)
rating_counts = rating_counts.loc[rating_counts.sum(axis=1).sort_values(ascending=False).index]

rating_counts.plot(kind="bar", stacked=True, color=[TYPE_COLORS[c] for c in rating_counts.columns], figsize=(11, 5))
plt.title("Titles by Rating")
plt.xlabel("Rating")
plt.ylabel("Number of titles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.4 Audience tier by content type (% within type)

In [ ]:
aud = (df.groupby(["type", "audience"]).size().unstack(fill_value=0))
aud_pct = aud.div(aud.sum(axis=1), axis=0) * 100
order = [c for c in ["Kids", "Older Kids", "Teens", "Adults", "Not Rated / Unknown"] if c in aud_pct.columns]

aud_pct[order].plot(kind="barh", stacked=True, figsize=(10, 3.5), colormap="Reds")
plt.title("Audience Tier by Content Type (%)")
plt.xlabel("% of titles")
plt.ylabel("")
plt.legend(title="Audience", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

aud_pct[order].round(1)

### 5.5 When titles are added: month and weekday

In [ ]:
dated = df.dropna(subset=["date_added"])
month_order = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
dated["month_added"].value_counts().reindex(month_order).plot(kind="bar", color=RED, ax=axes[0])
axes[0].set_title("Titles Added by Month")
axes[0].set_xlabel("")
axes[0].set_ylabel("Number of titles")

dated["day_added"].value_counts().reindex(day_order).plot(kind="bar", color=BLACK, ax=axes[1])
axes[1].set_title("Titles Added by Weekday")
axes[1].set_xlabel("")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

**Dashboard note:** Overview tab: type donut or bar, ratings bar (5.3). Optional "Deep dive" tab: duration histogram (5.1), seasons bar (5.2), audience mix (5.4), month/weekday bars (5.5).

## 6. Headline findings
The cell below computes the numbers from the data so the insights stay correct if the data changes. Copy the sentences you like into the dashboard's *Insights* section.

In [ ]:
total = len(df)
type_share = df["type"].value_counts(normalize=True) * 100

top_genre = genres["genre"].value_counts().index[0]
top_genre_n = genres["genre"].value_counts().iloc[0]

known_c = countries[countries["country_single"] != "Unknown"]
c_counts = known_c["country_single"].value_counts()

peak_added_year = int(df["year_added"].value_counts().idxmax())
peak_added_n = int(df["year_added"].value_counts().max())

recent_share = (df["release_year"] >= 2015).mean() * 100

lag_ok = df.dropna(subset=["release_lag"])
lag_ok = lag_ok[lag_ok["release_lag"] >= 0]

movies_ = df[df["type"] == "Movie"]["movie_minutes"].dropna()
shows_ = df[df["type"] == "TV Show"]["tv_seasons"].dropna()

findings = [
    f"1. The catalogue has {total:,} titles: {type_share.get('Movie', 0):.0f}% movies and {type_share.get('TV Show', 0):.0f}% TV shows.",
    f"2. The most common genre tag is '{top_genre}' ({top_genre_n:,} titles).",
    f"3. {c_counts.index[0]} leads production with {c_counts.iloc[0]:,} titles, followed by {c_counts.index[1]} ({c_counts.iloc[1]:,}) and {c_counts.index[2]} ({c_counts.iloc[2]:,}).",
    f"4. {recent_share:.0f}% of titles were released in 2015 or later.",
    f"5. The most titles were added in {peak_added_year} ({peak_added_n:,}).",
    f"6. The median gap between release and arrival on Netflix is {lag_ok['release_lag'].median():.0f} year(s).",
    f"7. The median movie runs {movies_.median():.0f} minutes; {(shows_ == 1).mean() * 100:.0f}% of TV shows have only one season.",
]

for f in findings:
    print(f)

### Findings to confirm by eye
Check these against the charts above and add your own interpretation:
- Which genres are growing or shrinking in the 2.3 trend chart?
- Is the non-US share in 4.4 rising?
- Do the TV show and movie genre mixes differ by country in 4.3?
- Any outliers in 5.1 (very short or very long movies) worth excluding from the dashboard?

### Chart shortlist for the dashboard
| Tab | Charts |
|---|---|
| Overview | KPIs, type split (2.x totals), ratings (5.3) |
| Genres | Top genres with type toggle (2.1, 2.2), trend (2.3) |
| Timeline | Release-year histogram (3.1), released vs added (3.3), lag (3.4) |
| Regions | Top countries (4.1), choropleth (new, Plotly), country-by-genre heatmap (4.3), non-US share (4.4) |
| Data explorer | Filterable table of `netflix_titles_cleaned.csv` |